In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('src')
from config.paths import config
from config.constants import TARGET_COLUMN, NON_MEDICAL_FEATURES

In [ ]:
df = pd.read_csv(config.cleaned_data_file)
print(f"Loaded dataset shape: {df.shape}")
print(f"Target distribution:\n{df[TARGET_COLUMN].value_counts()}")

In [ ]:
print("DATA VALIDATION CHECKS")
print("-" * 40)

def validate_data(df):
    validation_results = {}

    validation_results['total_rows'] = len(df)
    validation_results['total_columns'] = len(df.columns)
    validation_results['missing_values'] = df.isnull().sum().sum()

    validation_results['duplicate_rows'] = df.duplicated().sum()

    validation_results['target_distribution'] = df[TARGET_COLUMN].value_counts().to_dict()

    numerical_cols = df.select_dtypes(include=[np.number]).columns
    validation_results['numerical_outliers'] = {}
    for col in numerical_cols:
        if col != TARGET_COLUMN:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
            validation_results['numerical_outliers'][col] = outliers

    categorical_cols = df.select_dtypes(include=['object']).columns
    validation_results['high_cardinality'] = {}
    for col in categorical_cols:
        if df[col].nunique() > 20:
            validation_results['high_cardinality'][col] = df[col].nunique()

    return validation_results

validation_before = validate_data(df)
print("Validation Results - Before Preprocessing:")
print(f"Total rows: {validation_before['total_rows']}")
print(f"Total columns: {validation_before['total_columns']}")
print(f"Missing values: {validation_before['missing_values']}")
print(f"Duplicate rows: {validation_before['duplicate_rows']}")
print(f"Target distribution: {validation_before['target_distribution']}")

if validation_before['duplicate_rows'] > 0:
    df = df.drop_duplicates()
    print(f"Removed {validation_before['duplicate_rows']} duplicate rows")